In [10]:


def is_new_message(line):

    if len(line) < 8:
        return False

    return (
        line[0].isdigit()
        and line[1].isdigit()
        and line[2] == "/"
        and line[3].isdigit()
        and line[4].isdigit()
        and line[5] == "/"
        and line[6].isdigit()
        and line[7].isdigit()
    )


def is_deleted_message(message):

    return message.strip() == "This message was deleted"


def is_media_message(message):

    return "<Media omitted>" in message


def has_media_caption(message):

    cleaned = message.replace("<Media omitted>", "").strip()

    return len(cleaned) > 0


def parse_chat(filename):

    messages = []

    system_count = 0
    media_count = 0
    deleted_count = 0
    malformed_count = 0

    participants = set()

    current_message = None

    file = open(filename, "r", encoding="utf-8")

    for line in file:

        line = line.rstrip("\n")

        if line.strip() == "":
            continue

        if not is_new_message(line):

            if current_message is not None:
                current_message["message"] += "\n" + line

            continue

        if current_message is not None:
            messages.append(current_message)

        parts = line.split(" - ", 1)

        if len(parts) != 2:

            malformed_count += 1
            current_message = None
            continue

        timestamp = parts[0]
        remaining = parts[1]

        sender_parts = remaining.split(": ", 1)

        if len(sender_parts) != 2:

            system_count += 1
            current_message = None
            continue

        sender = sender_parts[0].strip()
        message = sender_parts[1].strip()

        participants.add(sender)

        message_type = "normal"

        if is_deleted_message(message):

            deleted_count += 1
            message_type = "deleted"

        elif is_media_message(message):

            media_count += 1

            if has_media_caption(message):
                message_type = "media_with_caption"
            else:
                message_type = "media_only"

        date_time = timestamp.split(", ", 1)

        if len(date_time) == 2:
            date = date_time[0]
            time = date_time[1]
        else:
            date = timestamp
            time = ""

        current_message = {

            "timestamp": timestamp,
            "date": date,
            "time": time,
            "sender": sender,
            "message": message,
            "type": message_type

        }

    if current_message is not None:
        messages.append(current_message)

    file.close()

    parser_summary = {

        "total_messages": len(messages),
        "participants": participants,
        "participant_count": len(participants),
        "system_messages": system_count,
        "media_messages": media_count,
        "deleted_messages": deleted_count,
        "malformed_messages": malformed_count

    }

    return messages, parser_summary


if __name__ == "__main__":

    messages, parser_summary = parse_chat("hostel_bois.txt")

    print("-"*3,"| GROUP OVERVIEW |","-"*30)

    print("Successfully Parsed Chat\n")

    print("Total Messages      :", parser_summary["total_messages"])
    print("Participants        :", parser_summary["participant_count"])
    print("System Messages     :", parser_summary["system_messages"])
    print("Media Messages      :", parser_summary["media_messages"])
    print("Deleted Messages    :", parser_summary["deleted_messages"])
    print("Malformed Messages  :", parser_summary["malformed_messages"])

    print("\nFirst Five Parsed Messages:\n")

    for message in messages[:5]:
        print(message)
        print()

    print("\nLast Five Parsed Messages:\n")

    for message in messages[-5:]:
        print(message)
        print()

--- | GROUP OVERVIEW | ------------------------------
Successfully Parsed Chat

Total Messages      : 3174
Participants        : 6
System Messages     : 4
Media Messages      : 32
Deleted Messages    : 15
Malformed Messages  : 0

First Five Parsed Messages:

{'timestamp': '01/04/24, 01:17', 'date': '01/04/24', 'time': '01:17', 'sender': 'Rahul', 'message': 'scene fix', 'type': 'normal'}

{'timestamp': '01/04/24, 01:17', 'date': '01/04/24', 'time': '01:17', 'sender': 'Rahul', 'message': 'haan', 'type': 'normal'}

{'timestamp': '01/04/24, 01:18', 'date': '01/04/24', 'time': '01:18', 'sender': 'Rahul', 'message': 'kya scene', 'type': 'normal'}

{'timestamp': '01/04/24, 02:13', 'date': '01/04/24', 'time': '02:13', 'sender': 'Rahul', 'message': 'abhi free hai?', 'type': 'normal'}

{'timestamp': '01/04/24, 02:13', 'date': '01/04/24', 'time': '02:13', 'sender': 'Rahul', 'message': 'abey', 'type': 'normal'}


Last Five Parsed Messages:

{'timestamp': '30/05/24, 19:14', 'date': '30/05/24', 'tim

In [11]:
from datetime import datetime

from parser import parse_chat


def format_date(date_string):

    date = datetime.strptime(date_string, "%d/%m/%y")

    return date.strftime("%d %B %Y")


def group_overview(messages, parser_summary):

    total_messages = parser_summary["total_messages"]

    participant_count = parser_summary["participant_count"]

    first_date = messages[0]["date"]

    last_date = messages[-1]["date"]

    first_date_object = datetime.strptime(first_date, "%d/%m/%y")

    last_date_object = datetime.strptime(last_date, "%d/%m/%y")

    total_days = (last_date_object - first_date_object).days + 1

    message_count = {}

    for message in messages:

        sender = message["sender"]

        if sender in message_count:

            message_count[sender] += 1

        else:

            message_count[sender] = 1

    sorted_people = sorted(

        message_count.items(),

        key=lambda person: person[1],

        reverse=True

    )

    print("=" * 60)
    print("GROUP OVERVIEW")
    print("=" * 60)

    print(f"Group            : Hostel Bois 4ever")

    print(
        f"Period           : {format_date(first_date)} to "
        f"{format_date(last_date)} ({total_days} days)"
    )

    print(f"Total Messages   : {total_messages}")

    print(f"Participants     : {participant_count}")

    print()

    print("MESSAGES PER PERSON")

    print()

    for sender, count in sorted_people:

        percentage = (count / total_messages) * 100

        print(f"{sender:<20} : {count:>5} ({percentage:5.1f}%)")


if __name__ == "__main__":

    messages, parser_summary = parse_chat("hostel_bois.txt")

    group_overview(messages, parser_summary)

GROUP OVERVIEW
Group            : Hostel Bois 4ever
Period           : 01 April 2024 to 30 May 2024 (60 days)
Total Messages   : 3174
Participants     : 6

MESSAGES PER PERSON

Rahul                :   953 ( 30.0%)
Priya                :   718 ( 22.6%)
Neha                 :   635 ( 20.0%)
Aman                 :   490 ( 15.4%)
Karan                :   354 ( 11.2%)
Vikas                :    24 (  0.8%)


In [12]:
from datetime import datetime

from parser import parse_chat


def format_date(date_string):

    date = datetime.strptime(date_string, "%d/%m/%y")

    return date.strftime("%d %B %Y")


def most_active_day_hour(messages):

    if len(messages) == 0:

        print("No messages found.")

        return

    day_count = {}

    hour_count = {}

    unique_days = set()

    for message in messages:

        date = message["date"]
        time = message["time"]

        unique_days.add(date)

        if date in day_count:

            day_count[date] += 1

        else:

            day_count[date] = 1

        if time != "":

            hour = time.split(":")[0]

            if hour in hour_count:

                hour_count[hour] += 1

            else:

                hour_count[hour] = 1

    if len(day_count) == 0:

        print("No valid dates found.")

        return

    if len(hour_count) == 0:

        print("No valid timestamps found.")

        return

    busiest_day = max(day_count, key=lambda day: day_count[day])

    busiest_day_messages = day_count[busiest_day]

    busiest_hour = max(hour_count, key=lambda hour: hour_count[hour])

    busiest_hour_messages = hour_count[busiest_hour]

    total_days = len(unique_days)

    average_messages = busiest_hour_messages / total_days

    print("=" * 60)
    print("                  MOST ACTIVE DAY AND HOUR")
    print("=" * 60)
    print()

    print(
        f"Busiest Day  : {format_date(busiest_day)} "
        f"({busiest_day_messages} messages)"
    )

    print(
        f"Busiest Hour : "
        f"{busiest_hour}:00 - "
        f"{int(busiest_hour) + 1:02d}:00 "
        f"(avg {average_messages:.1f} messages per day)"
    )


if __name__ == "__main__":

    messages, parser_summary = parse_chat("hostel_bois.txt")

    most_active_day_hour(messages)

                  MOST ACTIVE DAY AND HOUR

Busiest Day  : 04 May 2024 (76 messages)
Busiest Hour : 18:00 - 19:00 (avg 4.1 messages per day)


In [13]:
from datetime import datetime, timedelta

from parser import parse_chat


def average_response_time(messages):

    response_times = {}

    for message in messages:

        sender = message["sender"]

        if sender not in response_times:

            response_times[sender] = []

    for i in range(len(messages) - 1):

        first = messages[i]

        second = messages[i + 1]

        if first["sender"] == second["sender"]:

            continue

        first_time = datetime.strptime(
            first["timestamp"],
            "%d/%m/%y, %H:%M"
        )

        second_time = datetime.strptime(
            second["timestamp"],
            "%d/%m/%y, %H:%M"
        )

        gap = (second_time - first_time).total_seconds()

        response_times[second["sender"]].append(gap)

    average_times = {}

    for sender in response_times:

        gaps = response_times[sender]

        if len(gaps) == 0:

            average_times[sender] = None

        else:

            average_times[sender] = sum(gaps) / len(gaps)

    return average_times


def longest_silent_streak(messages):

    participants = set()

    active_days = {}

    first_date = datetime.strptime(
        messages[0]["date"],
        "%d/%m/%y"
    ).date()

    last_date = datetime.strptime(
        messages[-1]["date"],
        "%d/%m/%y"
    ).date()

    for message in messages:

        sender = message["sender"]

        participants.add(sender)

        if sender not in active_days:

            active_days[sender] = set()

        current_date = datetime.strptime(
            message["date"],
            "%d/%m/%y"
        ).date()

        active_days[sender].add(current_date)

    silent_streaks = {}

    for sender in participants:

        current_day = first_date

        current_streak = 0

        longest_streak = 0

        streak_start = None

        longest_start = None

        longest_end = None

        while current_day <= last_date:

            if current_day not in active_days[sender]:

                if current_streak == 0:

                    streak_start = current_day

                current_streak += 1

            else:

                if current_streak > longest_streak:

                    longest_streak = current_streak

                    longest_start = streak_start

                    longest_end = current_day - timedelta(days=1)

                current_streak = 0

            current_day += timedelta(days=1)

        if current_streak > longest_streak:

            longest_streak = current_streak

            longest_start = streak_start

            longest_end = last_date

        silent_streaks[sender] = (

            longest_streak,
            longest_start,
            longest_end

        )

    return silent_streaks


def format_time(seconds):

    if seconds is None:

        return "No replies"

    minutes = seconds / 60

    if minutes < 60:

        return f"{minutes:.1f} minutes"

    hours = minutes / 60

    return f"{hours:.1f} hours"


def print_response_report(average_times, silent_streaks):

    print()

    print("=" * 70)

    print("RESPONSE PATTERNS")

    print("=" * 70)

    print()

    valid = {

        person: average_times[person]

        for person in average_times

        if average_times[person] is not None

    }

    fastest = min(valid.items(), key=lambda item: item[1])[0]

    slowest = max(valid.items(), key=lambda item: item[1])[0]


    print(

        f"Fastest Replier : "

        f"{fastest} "

        f"({format_time(valid[fastest])})"

    )

    print(

        f"Slowest Replier : "

        f"{slowest} "

        f"({format_time(valid[slowest])})"

    )

    print()

    print("LONGEST SILENT STREAKS")

    print()

    ordered = sorted(

        silent_streaks.items(),

        key=lambda x: x[1][0],

        reverse=True

    )

    for sender, streak in ordered:

        days = streak[0]

        start = streak[1]

        end = streak[2]

        if days == 0:

            print(f"{sender:<12}: 0 days")

        elif days == 1:

            print(

                f"{sender:<12}: "

                f"1 day "

                f"({start.strftime('%d %b %Y')})"

            )

        else:

            print(

                f"{sender:<12}: "

                f"{days} days "

                f"({start.strftime('%d %b %Y')} "

                f"to "

                f"{end.strftime('%d %b %Y')})"

            )


if __name__ == "__main__":

    messages, parser_summary = parse_chat("hostel_bois.txt")

    average_times = average_response_time(messages)

    silent_streaks = longest_silent_streak(messages)

    print_response_report(

        average_times,

        silent_streaks

    )


RESPONSE PATTERNS

Fastest Replier : Rahul (34.9 minutes)
Slowest Replier : Aman (55.4 minutes)

LONGEST SILENT STREAKS

Vikas       : 11 days (23 Apr 2024 to 03 May 2024)
Karan       : 0 days
Neha        : 0 days
Rahul       : 0 days
Aman        : 0 days
Priya       : 0 days


In [14]:
import numpy as np

from parser import parse_chat


def activity_heatmap(messages):

    participants = []

    for message in messages:

        sender = message["sender"]

        if sender not in participants:

            participants.append(sender)

    participant_index = {}

    for i in range(len(participants)):

        participant_index[participants[i]] = i

    heatmap = np.zeros((len(participants), 24), dtype=int)

    for message in messages:

        sender = message["sender"]

        time = message["time"]

        if time == "":

            continue

        hour = int(time.split(":")[0])

        row = participant_index[sender]

        heatmap[row][hour] += 1

    return heatmap, participants

def print_ratio_matrix(heatmap, participants):

    print()
    print("=" * 130)
    print("NORMALIZED RATIO MATRIX")
    print("=" * 130)
    print()

    print(f"{'Participant / Time':<20}", end="")

    for hour in range(24):

        print(f"{hour:^7}", end="")

    print()

    print("-" * 190)

    for i in range(len(participants)):

        print(f"{participants[i]:<20}", end="")

        maximum = np.max(heatmap[i])

        if maximum == 0:

            maximum = 1

        for hour in range(24):

            ratio = heatmap[i][hour] / maximum

            print(f"{ratio:^7.2f}", end="")

        print()


def print_numeric_matrix(heatmap, participants):

    print()
    print("=" * 130)
    print("NUMERIC ACTIVITY MATRIX")
    print("=" * 130)
    print()

    print(f"{'Participant / Time':<20}", end="")

    for hour in range(24):

        print(f"{hour:^4}", end="")

    print()

    print("-" * 130)

    for i in range(len(participants)):

        print(f"{participants[i]:<20}", end="")

        for hour in range(24):

            print(f"{heatmap[i][hour]:^4}", end="")

        print()


def print_text_heatmap(heatmap, participants):

    print()
    print("=" * 130)
    print("TEXT ACTIVITY HEATMAP")
    print("=" * 130)
    print()

    print(f"{'Participant / Time':<20}", end="")

    for hour in range(24):

        print(f"{hour:^4}", end="")

    print()

    print("-" * 130)

    for i in range(len(participants)):

        print(f"{participants[i]:<20}", end="")

        maximum = np.max(heatmap[i])

        if maximum == 0:

            maximum = 1

        for hour in range(24):

            value = heatmap[i][hour]

            ratio = value / maximum

            if ratio <= 0.25:

                symbol = "\u00B7"

            elif ratio <= 0.50:

                symbol = "\u2591"

            elif ratio <= 0.75:

                symbol = "\u2592"

            else:

                symbol = "\u2588"

            print(f"{symbol:^4}", end="")

        print()


if __name__ == "__main__":

    messages, parser_summary = parse_chat("hostel_bois.txt")

    heatmap, participants = activity_heatmap(messages)

    print_numeric_matrix(heatmap, participants)

    print_ratio_matrix(heatmap, participants)

    print_text_heatmap(heatmap, participants)


NUMERIC ACTIVITY MATRIX

Participant / Time   0   1   2   3   4   5   6   7   8   9   10  11  12  13  14  15  16  17  18  19  20  21  22  23 
----------------------------------------------------------------------------------------------------------------------------------
Rahul                3   15  17  17  22  10  17  17  24  17  25  15  58  48  45  53  73  49 105  76  41  92  60  54 
Priya                0   0   0   0   0   0   13  20  47  65  62  61  57  48  44  29  32  40  38  60  43  32  18  9  
Karan                0   0   0   0   0   0   0   4   12  16  20  16  37  25  32  27  27  27  25  32  23  14  9   8  
Neha                 0   0   0   0   0   19  3   13  36  52  52  22  39  36  27  10  37  47  62  50  45  27  28  30 
Aman                 54  67  66  60  88  0   0   0   0   0   0   0   0   0   14  11  19  7   16  8   13  11  0   56 
Vikas                0   0   0   0   0   0   0   1   3   1   1   0   2   2   0   1   1   3   2   2   1   1   1   2  

NORMALIZED RATIO MATRIX

In [15]:
from parser import parse_chat


def top_words(messages):

    stop_words = {

        "i", "is", "the", "a", "an", "and", "or", "to", "of", "in",
        "on", "for", "at", "it", "this", "that", "was", "are", "am",
        "be", "been", "were", "you", "your", "me", "my", "we", "our",
        "us", "he", "she", "they", "them", "their", "as", "with",
        "from", "by", "if", "but", "so", "do", "did", "does", "have",
        "has", "had", "will", "would", "can", "could", "shall", "should"

    }

    punctuation = ".,!?;:'\"()[]{}<>-_/@#$%^&*=+`~|\\"

    word_count = {}

    for message in messages:

        if message["type"] == "deleted":

            continue

        if message["type"] == "media_only":

            continue

        text = message["message"].lower()

        words = text.split()

        for word in words:

            word = word.strip(punctuation)

            if word == "":

                continue

            if word in stop_words:

                continue

            if word not in word_count:

                word_count[word] = 1

            else:

                word_count[word] += 1

    sorted_words = sorted(

        word_count.items(),
        key=lambda item: item[1],
        reverse=True

    )

    return sorted_words[:10]


def print_top_words(top10):

    print()
    print("=" * 80)
    print("GROUPS FAVOURITE WORDS")
    print("=" * 80)
    print()

    maximum = top10[0][1]

    for word, count in top10:

        bar_length = int((count / maximum) * 30)

        bar = "\u2588" * bar_length

        print(f"{word:<20} {count:>5}  {bar}")
        print()



if __name__ == "__main__":

    messages, parser_summary = parse_chat("hostel_bois.txt")

    top10 = top_words(messages)

    print_top_words(top10)


GROUPS FAVOURITE WORDS

how                    321  ██████████████████████████████

guys                   318  █████████████████████████████

about                  274  █████████████████████████

hai                    268  █████████████████████████

today                  257  ████████████████████████

his                    217  ████████████████████

just                   208  ███████████████████

which                  202  ██████████████████

everyone               187  █████████████████

telling                179  ████████████████



In [16]:
from parser import parse_chat
from heatmap import activity_heatmap
from response_analysis import longest_silent_streak

from datetime import datetime

CARE_KEYWORDS = {

    "okay", "ok", "safe", "eat", "sleep",
    "take care", "please", "reminder",
    "drink water", "don't forget", "are you"

}

COMEDY_KEYWORDS = {

    "lol", "lmao", "haha", "rofl", "lmfao"

}

PROBLEM_SOLVER_KEYWORDS = {

    "try", "use", "install", "check",
    "restart", "update", "fix",
    "because", "instead", "replace",
    "change", "configure", "error",
    "issue", "solution", "problem",
    "debug"

}


def clean_words(text):

    punctuation = ".,!?;:'\"()[]{}<>-_/@#$%^&*=+`~|\\"

    words = []

    for word in text.lower().split():

        word = word.strip(punctuation)

        if word != "":

            words.append(word)

    return words


def initialise_scores(messages):

    scores = {}

    for message in messages:

        sender = message["sender"]

        if sender not in scores:

            scores[sender] = {}

    return scores




def detect_spammer(messages):

    spammer_scores = {}

    current_sender = messages[0]["sender"]

    current_burst = 1

    for message in messages:

        sender = message["sender"]

        if sender not in spammer_scores:

            spammer_scores[sender] = []

    for i in range(1, len(messages)):

        sender = messages[i]["sender"]

        previous_sender = messages[i - 1]["sender"]

        if sender == previous_sender:

            current_burst += 1

        else:

            spammer_scores[previous_sender].append(current_burst)

            current_burst = 1

    spammer_scores[messages[-1]["sender"]].append(current_burst)

    average_burst = {}

    for sender in spammer_scores:

        bursts = spammer_scores[sender]

        if len(bursts) == 0:

            average_burst[sender] = 0

        else:

            average_burst[sender] = sum(bursts) / len(bursts)

    return average_burst


def detect_group_mom(messages):

    caring_scores = {}

    total_messages = {}

    for message in messages:

        sender = message["sender"]

        if sender not in caring_scores:

            caring_scores[sender] = 0

            total_messages[sender] = 0

        total_messages[sender] += 1

        text = message["message"].lower()

        for keyword in CARE_KEYWORDS:

            if keyword in text:

                caring_scores[sender] += 1

                break

    percentages = {}

    for sender in caring_scores:

        if total_messages[sender] == 0:

            percentages[sender] = 0

        else:

            percentages[sender] = (

                caring_scores[sender]
                / total_messages[sender]

            ) * 100

    return percentages



def detect_night_owl(messages):

    night_messages = {}

    total_messages = {}

    for message in messages:

        sender = message["sender"]

        if sender not in night_messages:

            night_messages[sender] = 0

            total_messages[sender] = 0

        total_messages[sender] += 1

        if message["time"] == "":

            continue

        hour = int(message["time"].split(":")[0])

        if hour >= 23 or hour <= 4:

            night_messages[sender] += 1

    percentages = {}

    for sender in total_messages:

        if total_messages[sender] == 0:

            percentages[sender] = 0

        else:

            percentages[sender] = (

                night_messages[sender]

                / total_messages[sender]

            ) * 100

    return percentages


def detect_storyteller(messages):

    total_words = {}

    total_messages = {}

    for message in messages:

        sender = message["sender"]

        if sender not in total_words:

            total_words[sender] = 0

            total_messages[sender] = 0

        words = clean_words(message["message"])

        total_words[sender] += len(words)

        total_messages[sender] += 1

    average_words = {}

    for sender in total_words:

        if total_messages[sender] == 0:

            average_words[sender] = 0

        else:

            average_words[sender] = (

                total_words[sender]

                / total_messages[sender]

            )

    return average_words


def detect_drama_queen(messages):

    dramatic_messages = {}

    total_messages = {}

    for message in messages:

        sender = message["sender"]

        if sender not in dramatic_messages:

            dramatic_messages[sender] = 0

            total_messages[sender] = 0

        total_messages[sender] += 1

        text = message["message"]

        dramatic = False

        if len(text.strip()) >= 3:

            letters = ""

            for character in text:

                if character.isalpha():

                    letters += character

            if len(letters) > 0 and letters == letters.upper():

                dramatic = True

        if text.count("!") >= 2:

            dramatic = True

        if dramatic:

            dramatic_messages[sender] += 1

    percentages = {}

    for sender in dramatic_messages:

        if total_messages[sender] == 0:

            percentages[sender] = 0

        else:

            percentages[sender] = (

                dramatic_messages[sender]

                / total_messages[sender]

            ) * 100

    return percentages


def detect_ghost(messages):

    first_date = datetime.strptime(

        messages[0]["date"],

        "%d/%m/%y"

    ).date()

    last_date = datetime.strptime(

        messages[-1]["date"],

        "%d/%m/%y"

    ).date()

    total_days = (

        last_date - first_date

    ).days + 1

    silent_streaks = longest_silent_streak(messages)

    ghost_scores = {}

    for sender in silent_streaks:

        longest_streak = silent_streaks[sender][0]

        ghost_scores[sender] = (

            longest_streak

            / total_days

        ) * 100

    return ghost_scores



def detect_comedian(messages):

    comedy_scores = {}

    total_messages = {}

    for message in messages:

        sender = message["sender"]

        if sender not in comedy_scores:

            comedy_scores[sender] = 0

            total_messages[sender] = 0

        total_messages[sender] += 1

        words = clean_words(message["message"])

        for word in words:

            if word in COMEDY_KEYWORDS:

                comedy_scores[sender] += 1

                break

    percentages = {}

    for sender in comedy_scores:

        if total_messages[sender] == 0:

            percentages[sender] = 0

        else:

            percentages[sender] = (

                comedy_scores[sender]

                / total_messages[sender]

            ) * 100

    return percentages


def detect_question_master(messages):

    question_scores = {}

    total_messages = {}

    for message in messages:

        sender = message["sender"]

        if sender not in question_scores:

            question_scores[sender] = 0

            total_messages[sender] = 0

        total_messages[sender] += 1

        if message["message"].strip().endswith("?"):

            question_scores[sender] += 1

    percentages = {}

    for sender in question_scores:

        if total_messages[sender] == 0:

            percentages[sender] = 0

        else:

            percentages[sender] = (

                question_scores[sender]

                / total_messages[sender]

            ) * 100

    return percentages


def detect_problem_solver(messages):

    solver_scores = {}

    total_messages = {}

    for message in messages:

        sender = message["sender"]

        if sender not in solver_scores:

            solver_scores[sender] = 0

            total_messages[sender] = 0

        total_messages[sender] += 1

        words = clean_words(message["message"])

        for word in words:

            if word in PROBLEM_SOLVER_KEYWORDS:

                solver_scores[sender] += 1

                break

    percentages = {}

    for sender in solver_scores:

        if total_messages[sender] == 0:

            percentages[sender] = 0

        else:

            percentages[sender] = (

                solver_scores[sender]

                / total_messages[sender]

            ) * 100

    return percentages


def normalize_scores(archetype_scores):

    normalized_scores = {}

    for archetype in archetype_scores:

        normalized_scores[archetype] = {}

        maximum = max(archetype_scores[archetype].values())

        if maximum == 0:

            maximum = 1

        for sender in archetype_scores[archetype]:

            normalized_scores[archetype][sender] = (

                archetype_scores[archetype][sender]

                / maximum

            )

    return normalized_scores


def calculate_all_scores(messages):

    scores = {}

    scores["THE SPAMMER"] = detect_spammer(messages)

    scores["THE GROUP MOM"] = detect_group_mom(messages)

    scores["THE NIGHT OWL"] = detect_night_owl(messages)

    scores["THE STORYTELLER"] = detect_storyteller(messages)

    scores["THE DRAMA QUEEN"] = detect_drama_queen(messages)

    scores["THE GHOST"] = detect_ghost(messages)

    scores["THE COMEDIAN"] = detect_comedian(messages)

    scores["THE QUESTION MASTER"] = detect_question_master(messages)

    scores["THE PROBLEM SOLVER"] = detect_problem_solver(messages)

    return scores


'''def print_archetypes(assignments, raw_scores):

    print()

    print("=" * 70)

    print("PERSONALITY ARCHETYPES")

    print("=" * 70)

    print()

    ordered = sorted(assignments.items())

    for sender, archetype in ordered:

        value = raw_scores[archetype][sender]

        if archetype == "THE SPAMMER":

            metric = f"avg {value:.1f} msgs in a row"

        elif archetype == "THE GROUP MOM":

            metric = f"{value:.1f}% caring messages"

        elif archetype == "THE NIGHT OWL":

            metric = f"{value:.1f}% msgs after 11 PM"

        elif archetype == "THE STORYTELLER":

            metric = f"avg {value:.1f} words/msg"

        elif archetype == "THE DRAMA QUEEN":

            metric = f"{value:.1f}% dramatic msgs"

        elif archetype == "THE GHOST":

            metric = f"{value:.1f}% silent days"

        elif archetype == "THE COMEDIAN":

            metric = f"{value:.1f}% funny msgs"

        elif archetype == "THE QUESTION MASTER":

            metric = f"{value:.1f}% questions"

        elif archetype == "THE PROBLEM SOLVER":

            metric = f"{value:.1f}% solution msgs"

        else:

            metric = f"{value:.2f}"

        print(

            f"{sender:<10}"

            f"\u2192 "

            f"{archetype:<22}"

            f"({metric})"

        )
'''
def assign_archetypes(normalized_scores):

    participants = list(

        next(iter(normalized_scores.values())).keys()

    )

    preferences = {}

    for sender in participants:

        ranking = []

        for archetype in normalized_scores:

            score = normalized_scores[archetype][sender]

            ranking.append((archetype, score))

        ranking.sort(

            key=lambda item: item[1],

            reverse=True

        )

        preferences[sender] = ranking

    assignments = {}

    assigned_people = set()

    changed = True

    while changed:

        changed = False

        claims = {}

        for sender in participants:

            if sender in assigned_people:

                continue

            while len(preferences[sender]) > 0:

                archetype, score = preferences[sender][0]

                if archetype not in claims:

                    claims[archetype] = []

                claims[archetype].append(

                    (sender, score)

                )

                break

        for archetype in claims:

            candidates = claims[archetype]

            candidates.sort(

                key=lambda item: item[1],

                reverse=True

            )

            winner = candidates[0][0]

            assignments[winner] = archetype

            assigned_people.add(winner)

            changed = True

            for loser, score in candidates[1:]:

                preferences[loser].pop(0)

    return assignments


def print_archetypes(assignments, raw_scores):

    print()

    print("=" * 70)

    print("PERSONALITY ARCHETYPES")

    print("=" * 70)

    print()

    ordered = sorted(assignments.items())

    for sender, archetype in ordered:

        value = raw_scores[archetype][sender]

        if archetype == "THE SPAMMER":

            metric = f"avg {value:.1f} msgs in a row"

        elif archetype == "THE GROUP MOM":

            metric = f"{value:.1f}% caring messages"

        elif archetype == "THE NIGHT OWL":

            metric = f"{value:.1f}% msgs after 11 PM"

        elif archetype == "THE STORYTELLER":

            metric = f"avg {value:.1f} words/msg"

        elif archetype == "THE DRAMA QUEEN":

            metric = f"{value:.1f}% dramatic messages"

        elif archetype == "THE GHOST":

            metric = f"{value:.1f}% silent days"

        elif archetype == "THE COMEDIAN":

            metric = f"{value:.1f}% funny messages"

        elif archetype == "THE QUESTION MASTER":

            metric = f"{value:.1f}% questions"

        elif archetype == "THE PROBLEM SOLVER":

            metric = f"{value:.1f}% problem-solving messages"

        else:

            metric = f"{value:.2f}"

        print(f"{sender} \u2192 {archetype}    ({metric})")

if __name__ == "__main__":

    messages, parser_summary = parse_chat("hostel_bois.txt")

    raw_scores = calculate_all_scores(messages)

    normalized_scores = normalize_scores(raw_scores)

    assignments = assign_archetypes(normalized_scores)

    print_archetypes(assignments, raw_scores)


PERSONALITY ARCHETYPES

Aman → THE NIGHT OWL    (79.8% msgs after 11 PM)
Karan → THE STORYTELLER    (avg 55.7 words/msg)
Neha → THE DRAMA QUEEN    (62.2% dramatic messages)
Priya → THE GROUP MOM    (63.9% caring messages)
Rahul → THE SPAMMER    (avg 4.5 msgs in a row)
Vikas → THE GHOST    (18.3% silent days)


In [17]:
from analysis import group_overview

from activity_analysis import most_active_day_hour

from heatmap import (
    activity_heatmap,
    print_text_heatmap
)

from top_words import (
    top_words,
    print_top_words
)

from response_analysis import (
    average_response_time,
    longest_silent_streak,
    print_response_report
)

from archetype import (
    calculate_all_scores,
    normalize_scores,
    assign_archetypes,
    print_archetypes
)


def print_header():

    print("=" * 90)
    print(" " * 24 + "WHATSAPP GROUP ANALYTICS REPORT")
    print("=" * 90)
    print()


def print_footer():

    print()
    print("=" * 90)
    print("END OF REPORT")
    print("=" * 90)


if __name__ == "__main__":


    # Parser


    messages, parser_summary = parse_chat("hostel_bois.txt")

    print_header()


    # Group Overview


    group_overview(messages, parser_summary)

    print("\n")


    # Maximum Activity


    most_active_day_hour(messages)

    print("\n")


    # NumPy Heatmap


    heatmap, participants = activity_heatmap(messages)

    print_text_heatmap(heatmap, participants)

    print("\n")


    # Top Words


    top10 = top_words(messages)

    print_top_words(top10)

    print("\n")


    # Silent StreKS and Response Times


    average_times = average_response_time(messages)

    silent_streaks = longest_silent_streak(messages)

    print_response_report(
        average_times,
        silent_streaks
    )

    print("\n")


    # Personality Archetypes


    raw_scores = calculate_all_scores(messages)

    normalized_scores = normalize_scores(raw_scores)

    assignments = assign_archetypes(normalized_scores)

    print_archetypes(
        assignments,
        raw_scores
    )

    print_footer()

                        WHATSAPP GROUP ANALYTICS REPORT

GROUP OVERVIEW
Group            : Hostel Bois 4ever
Period           : 01 April 2024 to 30 May 2024 (60 days)
Total Messages   : 3174
Participants     : 6

MESSAGES PER PERSON

Rahul                :   953 ( 30.0%)
Priya                :   718 ( 22.6%)
Neha                 :   635 ( 20.0%)
Aman                 :   490 ( 15.4%)
Karan                :   354 ( 11.2%)
Vikas                :    24 (  0.8%)


                  MOST ACTIVE DAY AND HOUR

Busiest Day  : 04 May 2024 (76 messages)
Busiest Hour : 18:00 - 19:00 (avg 4.1 messages per day)



TEXT ACTIVITY HEATMAP

Participant / Time   0   1   2   3   4   5   6   7   8   9   10  11  12  13  14  15  16  17  18  19  20  21  22  23 
----------------------------------------------------------------------------------------------------------------------------------
Rahul                ·   ·   ·   ·   ·   ·   ·   ·   ·   ·   ·   ·   ▒   ░   ░   ▒   ▒   ░   █   ▒   ░   █   ▒   ▒  
Priy